# RQ1 Evaluation: Clustering Performance of LLM Label Discovery

This notebook evaluates the clustering ability of the LLM on the initial run (Phase 3: Dynamic Label Discovery) before metric learning or conversational refinement.

### Evaluation Setup
1. **Initial Clustering Workflow**: The LLM discovers clusters on a sampled subset of ArXiv abstracts ($N=100$) using Farthest Point Sampling (FPS) in mini-batches of 25.
2. **Aspects Evaluated**:
   * **Aspect 1: Topic/Category** (e.g. math, physics, computer science - Ground truth available)
   * **Aspect 2: Methodology** (e.g. theoretical proofs vs. computational/algorithmic simulations)
   * **Aspect 3: Application Domain** (e.g. industrial/real-world application vs. academic foundations)
3. **Metrics**:
   * **Ground-Truth Comparison**: For Aspect 1, compare LLM assignments against ground truth category labels using **Adjusted Rand Index (ARI)** and **Normalized Mutual Information (NMI)**.
   * **LLM-as-a-Judge Consistency Check**: For all three aspects, sample up to 20 representative documents from each cluster using Farthest Point Sampling (FPS) and prompt a Judge LLM to verify if they are consistent with their assigned cluster description. We calculate the purity of each cluster and report the **macro average** purity across all discovered clusters.


### Step 1: Set Up Environment & Imports


In [ ]:
import os
import sys
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure the root project path is in sys.path
sys.path.append(os.path.abspath('..'))

from src.dataset.loader import load_dataset
from src.embeddings.local_embeddings import SentenceTransformerEmbeddings
from src.pipeline import ConversationalClusteringPipeline
from src.agent.cloud_agent import GitHubModelsAgent, OpenAIAgent, GeminiAgent
from src.agent.local_agent import OllamaLocalAgent
from src.evaluation.metrics import compute_ari, compute_nmi, compute_acc

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Configurations
NUM_SAMPLES = 100
BATCH_SIZE = 25
RANDOM_STATE = 42

# GitHub Models API token fallback
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN", "")

print("Environment and imports set up successfully!")


### Step 2: Configure Generator & Judge Agents


In [ ]:
# Generator Agent (gpt-4o-mini with local Ollama fallback)
generator_agent = None
if GITHUB_TOKEN:
    generator_agent = GitHubModelsAgent(api_key=GITHUB_TOKEN, model_name="gpt-4o-mini", verbose=False)
elif os.getenv("OPENAI_API_KEY"):
    generator_agent = OpenAIAgent(model_name="gpt-4o-mini", verbose=False)
elif os.getenv("GEMINI_API_KEY"):
    generator_agent = GeminiAgent(model_name="gemini-1.5-flash", verbose=False)
else:
    generator_agent = OllamaLocalAgent(base_url="http://localhost:11434", model_name="qwen2.5-coder:latest", verbose=False)

# Judge Agent (gpt-4o with local Ollama fallback)
judge_agent = None
if GITHUB_TOKEN:
    judge_agent = GitHubModelsAgent(api_key=GITHUB_TOKEN, model_name="gpt-4o", verbose=False)
elif os.getenv("OPENAI_API_KEY"):
    judge_agent = OpenAIAgent(model_name="gpt-4o", verbose=False)
elif os.getenv("GEMINI_API_KEY"):
    judge_agent = GeminiAgent(model_name="gemini-1.5-pro", verbose=False)
else:
    judge_agent = OllamaLocalAgent(base_url="http://localhost:11434", model_name="qwen2.5-coder:latest", verbose=False)

print(f"Generator Agent: {generator_agent.__class__.__name__}")
print(f"Judge Agent: {judge_agent.__class__.__name__}")


### Step 3: Load Data & Precompute Embeddings


In [ ]:
dataset_path = "../data/arxiv_processed.json"
if not os.path.exists(dataset_path):
    dataset_path = "../data/arxiv_processed_sample.json"

dataset = load_dataset(dataset_path)
texts = dataset.get_texts()
true_categories = dataset.get_aspect_labels("category")

print(f"Loaded {len(texts)} documents.")

# Compute embeddings once
embedding_provider = SentenceTransformerEmbeddings()
raw_embeddings = embedding_provider.embed_texts(texts)
print(f"Generated base embeddings. Shape: {raw_embeddings.shape}")


### Step 4: Define Evaluation Helpers


In [ ]:
class PrecomputedEmbeddings:
    def __init__(self, embeddings):
        self.embeddings = embeddings
    def embed_texts(self, texts):
        return self.embeddings

def run_initial_discovery(user_intent, agent, raw_embeddings, texts):
    """
    Runs only Phase 2 (FPS) and Phase 3 (LLM Label Discovery) of the pipeline.
    """
    pre_provider = PrecomputedEmbeddings(raw_embeddings)
    pipeline = ConversationalClusteringPipeline(
        embedding_provider=pre_provider,
        agent=agent,
        num_samples=NUM_SAMPLES,
        batch_size=BATCH_SIZE,
        random_state=RANDOM_STATE,
        user_intent=user_intent,
        use_qa_phase=False
    )
    pipeline.set_data(texts)
    # Run Phase 3 Label Discovery
    pipeline._run_label_discovery(pipeline.sampled_indices, is_refinement=False)
    return pipeline.sampled_indices, pipeline.final_assignments, pipeline.global_registry

def sample_cluster_docs_fps(cluster_doc_indices, embeddings, num_samples=20):
    """
    Selects up to num_samples from cluster_doc_indices using Farthest Point Sampling (FPS).
    """
    if len(cluster_doc_indices) <= num_samples:
        return cluster_doc_indices
        
    sub_embs = embeddings[cluster_doc_indices]
    n = len(cluster_doc_indices)
    distances = np.full(n, np.inf)
    sampled_local_indices = []
    
    # Start with first document
    idx = 0
    sampled_local_indices.append(idx)
    
    for _ in range(1, num_samples):
        diff = sub_embs - sub_embs[idx]
        new_distances = np.sum(diff**2, axis=1)
        distances = np.minimum(distances, new_distances)
        idx = np.argmax(distances)
        sampled_local_indices.append(idx)
        
    return [cluster_doc_indices[i] for i in sampled_local_indices]

def parse_json_robustly(text: str) -> dict:
    content = text.strip()
    if content.startswith("```json"):
        content = content[7:]
    if content.startswith("```"):
        content = content[3:]
    if content.endswith("```"):
        content = content[:-3]
    content = content.strip()
    try:
        return json.loads(content)
    except Exception:
        match = re.search(r'\{.*\}', content, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except Exception:
                pass
        return {}

def evaluate_cluster_consistency(cluster_id, cluster_desc, doc_tuples, aspect_name, user_intent, judge):
    """
    Invokes the LLM Judge to evaluate the coherence of a single cluster.
    """
    if not doc_tuples:
        return []
        
    docs_str = ""
    for doc_id, text in doc_tuples:
        docs_str += f"--- Document ID: {doc_id} ---\n{text}\n\n"
        
    system_prompt = f"""You are an expert academic evaluator checking the quality of document clustering.
The clustering aspect is: \"{aspect_name}\"
The user's intent was: \"{user_intent}\"
The cluster description is: \"{cluster_desc}\" (ID: {cluster_id})

You are provided with a list of documents that were assigned to this cluster.
For each document, determine if it belongs to this cluster according to the cluster description and the main clustering intent.
- Answer 'true' (correct) if the document fits the description.
- Answer 'false' (incorrect) if the document does not fit or belongs to a different topic/theme.

You must respond ONLY with a valid JSON object matching this schema:
{{
  \"evaluations\": [
    {{
      \"doc_id\": <document ID as integer>,
      \"correct\": <true or false>,
      \"reason\": \"<short explanation>\"
    }},
    ...
  ]
}}
Do not write any markdown code blocks or additional text."""

    user_prompt = f"Please evaluate these documents:\n\n{docs_str}"
    
    try:
        response_text = judge.generate_text(system_prompt, user_prompt)
        parsed = parse_json_robustly(response_text)
        return parsed.get("evaluations", [])
    except Exception as e:
        print(f"Error judging cluster {cluster_id}: {e}")
        return [{"doc_id": doc_id, "correct": False, "reason": f"Error: {e}"} for doc_id, _ in doc_tuples]

print("Evaluation helpers defined successfully.")


### Step 5: Evaluate Aspect 1: Topic/Category (Ground Truth vs. LLM-as-a-Judge)


In [ ]:
aspect1_intent = "Group similar documents together by primary category for example math, physics, computer science, etc."
print("============================================================")
print("ASPECT 1: Topic/Category")
print("============================================================")

sampled_indices1, assignments1, registry1 = run_initial_discovery(
    user_intent=aspect1_intent,
    agent=generator_agent,
    raw_embeddings=raw_embeddings,
    texts=texts
)

print(f"Discovered {len(registry1)} clusters.")
for cid, desc in registry1.items():
    print(f"  - {cid}: {desc}")

# Method A: Compare with Ground Truth
unique_types = sorted(list(set(true_categories)))
label_map = {t: i for i, t in enumerate(unique_types)}
y_true = np.array([label_map[true_categories[idx]] for idx in sampled_indices1])

unique_pred_clusters = sorted(list(registry1.keys()))
pred_label_map = {c: i for i, c in enumerate(unique_pred_clusters)}
y_pred = np.array([pred_label_map[assignments1.get(idx, unique_pred_clusters[0])] for idx in sampled_indices1])

ari_val = compute_ari(y_true, y_pred)
nmi_val = compute_nmi(y_true, y_pred)
acc_val = compute_acc(y_true, y_pred)

print(f"\nMethod A: Ground Truth Comparison Metrics:")
print(f"  - Adjusted Rand Index (ARI): {ari_val:.4f}")
print(f"  - Normalized Mutual Information (NMI): {nmi_val:.4f}")
print(f"  - Clustering Accuracy (ACC): {acc_val:.4f}")

# Method B: LLM as a Judge
cluster_purities1 = {}
for cid, desc in registry1.items():
    cluster_docs = [idx for idx in sampled_indices1 if assignments1.get(idx) == cid]
    print(f"\nEvaluating Cluster {cid} ({len(cluster_docs)} docs assigned)...")
    if not cluster_docs:
        continue
        
    sampled_cluster_indices = sample_cluster_docs_fps(cluster_docs, raw_embeddings, num_samples=20)
    doc_tuples = [(idx, texts[idx]) for idx in sampled_cluster_indices]
    
    evals = evaluate_cluster_consistency(
        cluster_id=cid,
        cluster_desc=desc,
        doc_tuples=doc_tuples,
        aspect_name="Topic/Category",
        user_intent=aspect1_intent,
        judge=judge_agent
    )
    
    correct_count = sum(1 for e in evals if e.get("correct") is True)
    total_evaluated = len(evals)
    purity = correct_count / total_evaluated if total_evaluated > 0 else 0.0
    cluster_purities1[cid] = purity
    print(f"  - Cluster Purity: {purity:.2%} ({correct_count}/{total_evaluated})")

macro_purity1 = np.mean(list(cluster_purities1.values())) if cluster_purities1 else 0.0
print(f"\nAspect 1 Overall Macro-Averaged Purity: {macro_purity1:.2%}")


### Step 6: Evaluate Aspect 2: Methodology (LLM-as-a-Judge)


In [ ]:
aspect2_intent = "Group the documents by their methodology style: separate purely theoretical mathematical/proof papers from computational/simulation or software/algorithmic papers."
print("============================================================")
print("ASPECT 2: Methodology")
print("============================================================")

sampled_indices2, assignments2, registry2 = run_initial_discovery(
    user_intent=aspect2_intent,
    agent=generator_agent,
    raw_embeddings=raw_embeddings,
    texts=texts
)

print(f"Discovered {len(registry2)} clusters.")
for cid, desc in registry2.items():
    print(f"  - {cid}: {desc}")

# Method B: LLM as a Judge
cluster_purities2 = {}
for cid, desc in registry2.items():
    cluster_docs = [idx for idx in sampled_indices2 if assignments2.get(idx) == cid]
    print(f"\nEvaluating Cluster {cid} ({len(cluster_docs)} docs assigned)...")
    if not cluster_docs:
        continue
        
    sampled_cluster_indices = sample_cluster_docs_fps(cluster_docs, raw_embeddings, num_samples=20)
    doc_tuples = [(idx, texts[idx]) for idx in sampled_cluster_indices]
    
    evals = evaluate_cluster_consistency(
        cluster_id=cid,
        cluster_desc=desc,
        doc_tuples=doc_tuples,
        aspect_name="Methodology",
        user_intent=aspect2_intent,
        judge=judge_agent
    )
    
    correct_count = sum(1 for e in evals if e.get("correct") is True)
    total_evaluated = len(evals)
    purity = correct_count / total_evaluated if total_evaluated > 0 else 0.0
    cluster_purities2[cid] = purity
    print(f"  - Cluster Purity: {purity:.2%} ({correct_count}/{total_evaluated})")

macro_purity2 = np.mean(list(cluster_purities2.values())) if cluster_purities2 else 0.0
print(f"\nAspect 2 Overall Macro-Averaged Purity: {macro_purity2:.2%}")


### Step 7: Evaluate Aspect 3: Application Domain (LLM-as-a-Judge)


In [ ]:
aspect3_intent = "Group the documents by application domain: separate papers focused on industrial or real-world application domains from those focused on purely academic/theoretical foundations."
print("============================================================")
print("ASPECT 3: Application Domain")
print("============================================================")

sampled_indices3, assignments3, registry3 = run_initial_discovery(
    user_intent=aspect3_intent,
    agent=generator_agent,
    raw_embeddings=raw_embeddings,
    texts=texts
)

print(f"Discovered {len(registry3)} clusters.")
for cid, desc in registry3.items():
    print(f"  - {cid}: {desc}")

# Method B: LLM as a Judge
cluster_purities3 = {}
for cid, desc in registry3.items():
    cluster_docs = [idx for idx in sampled_indices3 if assignments3.get(idx) == cid]
    print(f"\nEvaluating Cluster {cid} ({len(cluster_docs)} docs assigned)...")
    if not cluster_docs:
        continue
        
    sampled_cluster_indices = sample_cluster_docs_fps(cluster_docs, raw_embeddings, num_samples=20)
    doc_tuples = [(idx, texts[idx]) for idx in sampled_cluster_indices]
    
    evals = evaluate_cluster_consistency(
        cluster_id=cid,
        cluster_desc=desc,
        doc_tuples=doc_tuples,
        aspect_name="Application Domain",
        user_intent=aspect3_intent,
        judge=judge_agent
    )
    
    correct_count = sum(1 for e in evals if e.get("correct") is True)
    total_evaluated = len(evals)
    purity = correct_count / total_evaluated if total_evaluated > 0 else 0.0
    cluster_purities3[cid] = purity
    print(f"  - Cluster Purity: {purity:.2%} ({correct_count}/{total_evaluated})")

macro_purity3 = np.mean(list(cluster_purities3.values())) if cluster_purities3 else 0.0
print(f"\nAspect 3 Overall Macro-Averaged Purity: {macro_purity3:.2%}")


### Step 8: Tabulate Results & Discussion


In [ ]:
# Aggregate Results
results_df = pd.DataFrame([
    {
        "Aspect": "Topic/Category",
        "Clusters Discovered": len(registry1),
        "ARI (vs Ground Truth)": f"{ari_val:.4f}",
        "NMI (vs Ground Truth)": f"{nmi_val:.4f}",
        "ACC (vs Ground Truth)": f"{acc_val:.4f}",
        "LLM Judge Purity (Macro Avg)": f"{macro_purity1:.2%}"
    },
    {
        "Aspect": "Methodology",
        "Clusters Discovered": len(registry2),
        "ARI (vs Ground Truth)": "N/A",
        "NMI (vs Ground Truth)": "N/A",
        "ACC (vs Ground Truth)": "N/A",
        "LLM Judge Purity (Macro Avg)": f"{macro_purity2:.2%}"
    },
    {
        "Aspect": "Application Domain",
        "Clusters Discovered": len(registry3),
        "ARI (vs Ground Truth)": "N/A",
        "NMI (vs Ground Truth)": "N/A",
        "ACC (vs Ground Truth)": "N/A",
        "LLM Judge Purity (Macro Avg)": f"{macro_purity3:.2%}"
    }
])

print("=======================================================================")
print("                          SUMMARY METRICS TABLE                        ")
print("=======================================================================")
print(results_df.to_string(index=False))
print("=======================================================================")

# Plot Purity Chart
plt.figure(figsize=(7, 4.5))
aspects = ["Topic/Category", "Methodology", "Application Domain"]
purities = [macro_purity1, macro_purity2, macro_purity3]
colors = ["#4c72b0", "#55a868", "#c44e52"]

plt.bar(aspects, purities, color=colors, alpha=0.85, width=0.4)
plt.ylabel("LLM Judge Macro-Averaged Purity")
plt.title("LLM Discovery Phase Consistency Purity across 3 Aspects")
plt.ylim(0, 1.15)
for i, v in enumerate(purities):
    plt.text(i, v + 0.02, f"{v:.2%}", ha="center", fontweight="bold")
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()

os.makedirs("../Output", exist_ok=True)
plt.savefig("../Output/rq1_accuracy_chart.png", dpi=150)
plt.show()

# Export JSON results
summary_dict = {
    "topic_category": {
        "clusters_discovered": len(registry1),
        "ari": ari_val,
        "nmi": nmi_val,
        "acc": acc_val,
        "judge_purity": macro_purity1
    },
    "methodology": {
        "clusters_discovered": len(registry2),
        "judge_purity": macro_purity2
    },
    "application_domain": {
        "clusters_discovered": len(registry3),
        "judge_purity": macro_purity3
    }
}

with open("../Output/rq1_results.json", "w") as f:
    json.dump(summary_dict, f, indent=4)
print("Metrics exported successfully to Output/rq1_results.json")
